# KarmaLego Experiments

End-to-end temporal pattern mining pipeline covering:
- **Section 1 – Discover Patterns**
- **Section 2 – Apply Patterns to Patients**

> Launch Jupyter from the `KarmaLego/` root directory so that `core` is importable.  
> Run the **Setup** cell once per environment to install dependencies.

---
## Setup & Imports

Install dependencies and import all required modules.

In [ ]:
# # Install the package and its dependencies (run once per environment)
# !pip install -e . -q

In [12]:
import os
import pandas as pd
from ast import literal_eval

from core.karmalego import KarmaLego, TIRP
from core.io import (
    validate_input,
    build_or_load_mappings,
    preprocess_dataframe,
    to_entity_list,
)

print("All imports OK")

All imports OK


---
## Section 1 – Discover Patterns

### 1.1 – Data Loading & Validation

Load the input CSV and verify that required columns are present.

In [4]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATA_CSV      = "data/synthetic_diabetes_temporal_data.csv"
MAPPING_DIR   = "data"
PATTERNS_PATH = "data/discovered_patterns.csv"
OUTPUT_PATH   = "data/patient_pattern_vectors.ALL.csv"

# For large files swap to: import dask.dataframe as dd; df = dd.read_csv(DATA_CSV)
df = pd.read_csv(DATA_CSV)

validate_input(df)
print(f"Loaded {len(df):,} rows — schema OK")
df.head()

Loaded 1,312 rows — schema OK


,PatientId,ConceptName,StartDateTime,EndDateTime,Value
0,1002,GLUCOSE_MEASURE_STATE,1/2/2025 16:00,1/2/2025 20:00,Low Glucose
1,1002,ADMISSION_EVENT,1/2/2025 16:00,1/2/2025 16:00,TRUE
2,1002,GLUCOSE_MEASURE_TREND,1/2/2025 16:00,1/3/2025 0:00,Increasing
3,1002,DIABETES_DIAGNOSIS_CONTEXT,1/2/2025 16:00,1/6/2025 22:00,TRUE
4,1002,MEAL_CONTEXT,1/2/2025 17:00,1/2/2025 21:00,Dinner


### 1.2 – Symbol Mapping & Preprocessing

Build (or reuse) a symbol map, encode symbols, and convert to the in-memory entity list.

In [5]:
symbol_map, inverse_symbol_map = build_or_load_mappings(df, mapping_dir=MAPPING_DIR, reuse=True)
print(f"Symbol map: {len(symbol_map)} unique concept:value pairs")

Symbol map: 23 unique concept:value pairs


In [6]:
preprocessed = preprocess_dataframe(df, symbol_map)
entity_list, patient_ids = to_entity_list(preprocessed)
print(f"Entities: {len(entity_list)} | Patients: {len(patient_ids)}")

Entities: 10 | Patients: 10


### 1.3 – Run KarmaLego

Instantiate `KarmaLego` and discover frequent TIRPs.  
Results are cached to `discovered_patterns.csv`; subsequent runs reload from disk.

| Parameter | Meaning |
|-----------|---------|
| `epsilon` | Temporal tolerance for equality / meet decisions |
| `max_distance` | Max gap between two intervals to still consider them related |
| `min_ver_supp` | Minimum fraction of entities that must support a pattern |
| `num_relations` | Relation granularity: 2 / 3 / 5 / 7 (Allen algebra levels) |

In [8]:
kl = KarmaLego(
    epsilon=pd.Timedelta(seconds=2),
    max_distance=pd.Timedelta(hours=4),
    min_ver_supp=0.5,
    num_relations=7,
)

if os.path.exists(PATTERNS_PATH):
    print(f"Loading patterns from {PATTERNS_PATH} …")
    patterns_df = pd.read_csv(PATTERNS_PATH)
else:
    print("Running KarmaLego discovery …")
    patterns_df = kl.discover_patterns(entity_list, min_length=1)
    patterns_df.to_csv(PATTERNS_PATH, index=False)
    print(f"Saved {len(patterns_df)} patterns → {PATTERNS_PATH}")

print(f"Discovered patterns: {len(patterns_df)}")
patterns_df.head()

INFO:core.karmalego:Starting discover_patterns


Running KarmaLego discovery …


Karma (pairs):  99%|█████████▉| 9498/9617 [00:00<00:00, 1895669.73it/s]
Lego phase (nodes expanded): 1696 node/s [00:13, 125.19 node/s/s]
INFO:core.karmalego:discover_patterns timings (min): pre=0.0000 karma=0.0005 lego=0.2260 flatten=0.0002 total=0.2267
INFO:core.karmalego:Finished discover_patterns


Saved 1696 patterns → data/discovered_patterns.csv
Discovered patterns: 1696


,symbols,relations,k,vertical_support,tirp_obj,tirp_str,is_closed,is_super_pattern
0,"(22,)",(),1,1.0,None,RELEASE_EVENT:TRUE,False,False
1,"(21,)",(),1,1.0,None,MEAL_CONTEXT:Lunch,False,False
2,"(20,)",(),1,1.0,None,MEAL_CONTEXT:Dinner,False,False
3,"(19,)",(),1,1.0,None,MEAL_CONTEXT:Breakfast,False,False
4,"(18,)",(),1,1.0,None,GLUCOSE_MEASURE_TREND:Steady,False,False


In [9]:
# Reconstruct lightweight TIRP objects from the saved symbols / relations columns
sym_series = patterns_df["symbols"].apply(
    lambda x: x if not isinstance(x, str) else literal_eval(x)
)
rel_series = patterns_df["relations"].apply(
    lambda x: x if not isinstance(x, str) else literal_eval(x)
)

patterns_tirps = [
    TIRP(
        epsilon=kl.epsilon,
        max_distance=kl.max_distance,
        min_ver_supp=kl.min_ver_supp,
        symbols=list(syms),
        relations=list(rels),
        k=len(syms),
    )
    for syms, rels in zip(sym_series, rel_series)
]

rep_to_str   = {(tuple(t.symbols), tuple(t.relations)): s
                for t, s in zip(patterns_tirps, patterns_df["tirp_str"])}
pattern_keys = [(tuple(t.symbols), tuple(t.relations)) for t in patterns_tirps]

print(f"Reconstructed {len(patterns_tirps)} TIRP objects")

Reconstructed 1696 TIRP objects


---
## Section 2 – Apply Patterns to Patients

Compute all five feature columns per (patient, pattern) pair.

| Column | Mode | Strategy | Meaning |
|--------|------|----------|---------|
| `tirp_count_unique_last` | `tirp-count` | `unique_last` | One count per distinct last-symbol index |
| `tirp_count_all` | `tirp-count` | `all` | Every valid embedding counted |
| `tpf_dist_unique_last` | `tpf-dist` | `unique_last` | Min-max of `tirp_count_unique_last` across cohort |
| `tpf_dist_all` | `tpf-dist` | `all` | Min-max of `tirp_count_all` across cohort |
| `tpf_duration` | `tpf-duration` | `unique_last` | Union of embedding time-spans, min-max normalised |

In [10]:
print("Computing tirp-count (unique_last) …")
vec_count_ul = kl.apply_patterns_to_entities(
    entity_list, patterns_tirps, patient_ids,
    mode="tirp-count", count_strategy="unique_last"
)

print("Computing tirp-count (all) …")
vec_count_all = kl.apply_patterns_to_entities(
    entity_list, patterns_tirps, patient_ids,
    mode="tirp-count", count_strategy="all"
)

print("Computing tpf-dist (unique_last) …")
vec_tpf_dist_ul = kl.apply_patterns_to_entities(
    entity_list, patterns_tirps, patient_ids,
    mode="tpf-dist", count_strategy="unique_last"
)

print("Computing tpf-dist (all) …")
vec_tpf_dist_all = kl.apply_patterns_to_entities(
    entity_list, patterns_tirps, patient_ids,
    mode="tpf-dist", count_strategy="all"
)

print("Computing tpf-duration …")
vec_tpf_duration = kl.apply_patterns_to_entities(
    entity_list, patterns_tirps, patient_ids,
    mode="tpf-duration", count_strategy="unique_last"
)

print("Done.")

INFO:core.karmalego:Starting apply_patterns_to_entities


Computing tirp-count (unique_last) …


Applying patterns to entities: 100%|██████████| 1696/1696 [00:01<00:00, 1238.15it/s]
INFO:core.karmalego:Finished apply_patterns_to_entities
INFO:core.karmalego:Starting apply_patterns_to_entities


Computing tirp-count (all) …


Applying patterns to entities: 100%|██████████| 1696/1696 [00:01<00:00, 1228.90it/s]
INFO:core.karmalego:Finished apply_patterns_to_entities
INFO:core.karmalego:Starting apply_patterns_to_entities


...

Computing tpf-dist (all) …


Applying patterns to entities: 100%|██████████| 1696/1696 [00:01<00:00, 1237.62it/s]
INFO:core.karmalego:Finished apply_patterns_to_entities
INFO:core.karmalego:Starting apply_patterns_to_entities


Computing tpf-duration …


Applying patterns to entities: 100%|██████████| 1696/1696 [00:01<00:00, 1261.73it/s]
INFO:core.karmalego:Finished apply_patterns_to_entities


Done.


In [11]:
rows = []
for pid in patient_ids:
    for rep in pattern_keys:
        rows.append({
            "PatientID":              pid,
            "Pattern":                rep_to_str.get(rep, str(rep)),
            "tirp_count_unique_last": vec_count_ul.get(pid, {}).get(rep, 0.0),
            "tirp_count_all":         vec_count_all.get(pid, {}).get(rep, 0.0),
            "tpf_dist_unique_last":   vec_tpf_dist_ul.get(pid, {}).get(rep, 0.0),
            "tpf_dist_all":           vec_tpf_dist_all.get(pid, {}).get(rep, 0.0),
            "tpf_duration":           vec_tpf_duration.get(pid, {}).get(rep, 0.0),
        })

out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(out_df):,} rows → {OUTPUT_PATH}")
out_df.head(10)

Saved 16,960 rows → data/patient_pattern_vectors.ALL.csv


,PatientID,Pattern,tirp_count_unique_last,tirp_count_all,tpf_dist_unique_last,tpf_dist_all,tpf_duration
0,1000,RELEASE_EVENT:TRUE,1.0,1.0,0.000000,0.000000,0.000000
1,1000,MEAL_CONTEXT:Lunch,4.0,4.0,0.100000,0.100000,0.076923
2,1000,MEAL_CONTEXT:Dinner,3.0,3.0,0.000000,0.000000,0.000000
3,1000,MEAL_CONTEXT:Breakfast,3.0,3.0,0.000000,0.000000,0.000000
4,1000,GLUCOSE_MEASURE_TREND:Steady,6.0,6.0,0.062500,0.062500,0.000000
5,1000,GLUCOSE_MEASURE_TREND:Increasing,3.0,3.0,0.000000,0.000000,0.000000
6,1000,GLUCOSE_MEASURE_TREND:Decreasing,6.0,6.0,0.230769,0.230769,0.167173
7,1000,GLUCOSE_MEASURE_STATE:Normal Glucose,3.0,3.0,0.111111,0.111111,0.000000
8,1000,GLUCOSE_MEASURE_STATE:Low Glucose,3.0,3.0,0.076923,0.076923,0.131455
9,1000,GLUCOSE_MEASURE_STATE:High Glucose,2.0,2.0,0.000000,0.000000,0.000000
